# OmniShield-X trên Kaggle
Đính kèm gói `omnishield-src.zip`, chọn GPU cho job train, rồi đặt `JOB` và các đường dẫn bên dưới. Các notebook chạy độc lập. Dùng thư mục run mới cho mỗi lần chạy.

In [ ]:
from pathlib import Path

JOB = 'prepare-mvp'  # prepare-mvp, mvp-real, replay-mvp, smoke, prepare, edge-42, edge-77, edge-123, final-supervised, evaluation
SOURCE_ZIP = Path('/kaggle/input/omnishield-source/omnishield-src.zip')
DATA_ZIP = None  # Optional Path('/kaggle/input/phenikaa-mvp/phenikaa-mvp.zip')
RAW = Path('/kaggle/input/phenikaa-mvp/raw')
PREPARED = Path('/kaggle/input/phenikaa-mvp/prepared')
CALIBRATION = RAW / 'calib information'
RIG_CONFIG = Path('/kaggle/input/omnishield-rig/phenikaa.json')
DINO_SOURCE = Path('/kaggle/input/dinov3-source/dinov3')
DINO_WEIGHTS = Path('/kaggle/input/dinov3-weights/dinov3_vits16.pth')
CHECKPOINT = None  # Path to last.pt for resume, or best.pt for evaluation
INITIAL_CHECKPOINT = None  # best.pt from the Edge run for a new final run
SPLIT = 'val'
MAX_MINUTES = 240
PRECISION = 'fp16'
MVP_FRAMES_PER_CLIP = None  # None selects all verified frames. Use 4 for a data-only diagnostic.
MVP_REPLAY_FRAMES = 80

In [ ]:
import hashlib, json, shutil, subprocess, sys
from zipfile import ZipFile

def unpack(source, destination):
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Invalid archive member: {member.filename}')
            if not member.is_dir():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))

def resolve_input(path):
    if path is None or path.exists():
        return path
    parts = path.parts
    if len(parts) >= 4 and parts[:3] == ('/', 'kaggle', 'input') and parts[3] != 'datasets':
        slug, rest = parts[3], Path(*parts[4:])
        for root in (Path('/kaggle/input/datasets'), Path('/kaggle/input/notebooks')):
            for base in root.glob(f'*/{slug}'):
                candidate = base / rest
                if candidate.exists():
                    return candidate
    return path

SOURCE_ZIP = resolve_input(SOURCE_ZIP)
source_dir = resolve_input(SOURCE_ZIP.parent)
if SOURCE_ZIP.is_file():
    digest = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
    project = Path('/kaggle/working') / f'source-{digest[:12]}'
    unpack(SOURCE_ZIP, project)
elif (source_dir / 'omnishield' / '__main__.py').is_file():
    hasher = hashlib.sha256()
    for item in sorted(p for p in source_dir.rglob('*') if p.is_file()):
        hasher.update(item.relative_to(source_dir).as_posix().encode())
        hasher.update(item.read_bytes())
    digest = hasher.hexdigest()
    project = Path('/kaggle/working') / f'source-{digest[:12]}'
    shutil.copytree(source_dir, project, dirs_exist_ok=True)
else:
    found = [str(p) for p in sorted(Path('/kaggle/input').glob('*'))] if Path('/kaggle/input').exists() else []
    raise FileNotFoundError(f'Attach the source ZIP and update SOURCE_ZIP: {SOURCE_ZIP} (found {found})')

job = json.loads((project / 'kaggle/jobs.json').read_text())[JOB]
out = Path('/kaggle/working/runs') / job['id']

RAW, PREPARED = resolve_input(RAW), resolve_input(PREPARED)
DINO_SOURCE, DINO_WEIGHTS = resolve_input(DINO_SOURCE), resolve_input(DINO_WEIGHTS)
CHECKPOINT, INITIAL_CHECKPOINT = resolve_input(CHECKPOINT), resolve_input(INITIAL_CHECKPOINT)
if DATA_ZIP is not None and job['mode'] in {'train', 'score', 'replay-mvp'}:
    DATA_ZIP = resolve_input(DATA_ZIP)
    mounted = Path('/kaggle/working/phenikaa-mvp-input')
    unpack(DATA_ZIP, mounted)
    RAW, PREPARED = mounted / 'raw', mounted / 'prepared'

if job['mode'] in {'train', 'score', 'replay-mvp'} and not (DINO_SOURCE / 'hubconf.py').is_file():
    dino_root = resolve_input(DINO_SOURCE.parent)
    if (dino_root / 'hubconf.py').is_file():
        DINO_SOURCE = dino_root

def run(*args):
    subprocess.run([sys.executable, '-m', 'omnishield', *map(str, args)], cwd=project, check=True)

print('Source SHA-256:', digest, '| job:', JOB, '| output:', out)

In [ ]:
mode = job['mode']
if mode in {'smoke', 'train', 'replay-mvp', 'score'}:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the GPU accelerator for training.')
    print(torch.__version__, torch.cuda.get_device_name(0))

if mode in {'prepare', 'prepare-mvp'}:
    import importlib.util
    if not all(importlib.util.find_spec(package) for package in ('laspy', 'lazrs')):
        raise RuntimeError('Install laspy[lazrs] in Kaggle, or supply wheels as a Notebook input.')
    raw, calibration, rig = RAW, CALIBRATION, RIG_CONFIG
    if mode == 'prepare-mvp':
        data = Path('/kaggle/working/phenikaa-mvp')
        raw, prepared = data / 'raw', data / 'prepared'
        calibration, rig = raw / 'calibration', 'configs/rig/phenikaa_mvp.json'
        model = 'configs/model/edge_mvp.json'
        limit = [] if MVP_FRAMES_PER_CLIP is None else ['--max-frames-per-clip', MVP_FRAMES_PER_CLIP]
        run('acquire-mvp', '--raw', raw, *limit)
    else:
        prepared, model = Path('/kaggle/working/phenikaa-prepared'), 'configs/model/edge.json'
    run('ingest', '--raw', raw, '--calibration', calibration, '--rig', rig, '--out', prepared / 'manifest.json')
    run('calibrate', '--prepared', prepared, '--model', model)
    run('labels', '--raw', raw, '--prepared', prepared, '--model', model)
    run('audit', '--raw', raw, '--prepared', prepared, '--model', model)
    run('render-replay', '--raw', raw, '--prepared', prepared, '--model', model, '--out', out / 'data-review.png')
elif mode == 'smoke':
    raw = Path('/kaggle/working/fixture-rgb')
    prepared = Path('/kaggle/working/fixture-labels')
    run('fixture', '--raw', raw, '--prepared', prepared)
    run('train', '--raw', raw, '--prepared', prepared, '--config', 'configs/train/smoke.json',
        '--allow-synthetic', '--out', out, '--workers', 0, '--amp', 'off')
elif mode == 'train':
    continuation = []
    if CHECKPOINT is not None and INITIAL_CHECKPOINT is not None:
        raise ValueError('Choose either continuation or fine-tune initialization.')
    if CHECKPOINT is not None:
        continuation = ['--resume', CHECKPOINT]
    elif INITIAL_CHECKPOINT is not None:
        continuation = ['--init', INITIAL_CHECKPOINT]
    else:
        continuation = ['--weights', DINO_WEIGHTS]
    run('train', '--raw', RAW, '--prepared', PREPARED, '--config', job['config'], '--dino-source', DINO_SOURCE,
        '--out', out, '--seed', job['seed'], '--max-minutes', MAX_MINUTES, '--amp', PRECISION, *continuation)
elif mode in {'score', 'replay-mvp'}:
    if CHECKPOINT is None:
        raise ValueError('Set CHECKPOINT to the best.pt for the run.')
    shared = ['--raw', RAW, '--prepared', PREPARED, '--checkpoint', CHECKPOINT, '--split', SPLIT,
              '--device', 'cuda', '--dino-source', DINO_SOURCE]
    run('score', *shared, '--out', out / 'metrics.json')
    if mode == 'replay-mvp':
        run('predict', *shared, '--out', out / 'prediction')
        run('render-replay', '--raw', RAW, '--prepared', PREPARED, '--split', SPLIT,
            '--model', 'configs/model/edge_mvp.json', '--pred', out / 'prediction',
            '--frames', MVP_REPLAY_FRAMES, '--out', out / 'rgb-replay.gif')
else:
    raise ValueError(f'Unknown job mode: {mode}')